# 2013 Round 1 Acquisition Financing - Debt Facility Model

This notebook builds a complete financial model for a 4-tranche debt facility,
computing amortization schedules, interest payments, and answering all 6 questions.

In [ ]:
import numpy as np
from datetime import date

# ============================================================
# HELPER: Generate quarterly end dates
# ============================================================
def generate_quarter_ends(start_year, start_quarter, end_year, end_quarter):
    """Generate list of quarter-end dates (31 Mar, 30 Jun, 30 Sep, 31 Dec)."""
    dates = []
    q_ends = {1: (3, 31), 2: (6, 30), 3: (9, 30), 4: (12, 31)}
    y, q = start_year, start_quarter
    while (y, q) <= (end_year, end_quarter):
        m, d = q_ends[q]
        dates.append(date(y, m, d))
        q += 1
        if q > 4:
            q = 1
            y += 1
    return dates

# Generate dates from Q4 2013 to Q4 2023 (covers all maturities)
all_dates = generate_quarter_ends(2013, 4, 2023, 4)
print(f"Generated {len(all_dates)} quarter-end dates from {all_dates[0]} to {all_dates[-1]}")

In [ ]:
# ============================================================
# TRANCHE 1: Credit Foncier (Equal P+I Payments)
# ============================================================
# $14m drawdown on 31 Dec 2013
# 7 years = 28 quarters
# 8.25% p.a. simple rate
# 30/360 day count => quarterly rate = annual_rate / 4
# Amortizes quarterly in arrears on credit foncier schedule
# (equal total P+I payments each quarter)

t1_drawdown = 14_000_000
t1_start = date(2013, 12, 31)
t1_maturity = date(2020, 12, 31)
t1_rate = 0.0825
t1_qrate = t1_rate / 4  # 2.0625%
t1_nperiods = 28  # 7 years * 4 quarters

# Credit foncier PMT = PV * r / (1 - (1+r)^(-n))
t1_pmt = t1_drawdown * t1_qrate / (1 - (1 + t1_qrate)**(-t1_nperiods))
print(f"Tranche 1 quarterly PMT: ${t1_pmt:,.2f}")

# Build amortization schedule
t1_schedule = []
balance = 0
for dt in all_dates:
    if dt < t1_start:
        continue
    if dt == t1_start:
        t1_schedule.append({
            'date': dt, 'opening': 0, 'drawdown': t1_drawdown,
            'interest_expense': 0, 'interest_paid': 0,
            'interest_capitalized': 0, 'principal_paid': 0,
            'closing': t1_drawdown
        })
        balance = t1_drawdown
        continue
    if dt > t1_maturity:
        break
    opening = balance
    interest = opening * t1_qrate
    principal = t1_pmt - interest
    balance = opening - principal
    t1_schedule.append({
        'date': dt, 'opening': opening, 'drawdown': 0,
        'interest_expense': interest, 'interest_paid': interest,
        'interest_capitalized': 0, 'principal_paid': principal,
        'closing': balance
    })

print(f"Tranche 1 schedule has {len(t1_schedule)} entries")
print(f"Tranche 1 final closing balance: ${t1_schedule[-1]['closing']:,.6f}")

# Q2 check: closing balance on 31 Mar 2020
for row in t1_schedule:
    if row['date'] == date(2020, 3, 31):
        print(f"Tranche 1 closing on 31 Mar 2020: ${row['closing']:,.2f} = ${row['closing']/1e6:.4f}m")

In [ ]:
# ============================================================
# TRANCHE 2: IO then Level Principal Amortization
# ============================================================
# $25m drawdown on 31 Dec 2013
# 9 years = 36 quarters to maturity (31 Dec 2022)
# Interest only for first 5 years at 9.00% p.a.
# Then amortizes with level principal at 8.50% p.a.
# First principal repayment 5.25 years after drawdown
#   5.25 years = 21 quarters after drawdown = quarter 21 = 31 Mar 2019
# From quarter 21 to quarter 36 = 16 amortization quarters
# Level principal = $25m / 16 = $1,562,500
#
# IMPORTANT: The IO period is 5 years (20 quarters of payments)
# but the interest rate changes at 5 years (quarter 21 onwards is amort rate)
# Quarter 21 is the first quarter where principal is paid.
# The problem says: "interest only for the first 5 years, and thereafter
# amortizes quarterly in arrears via level Principal repayments
# (the first principal repayment is 5.25 years after drawdown)"
# So: quarters 1-20 are IO at 9%, quarter 21 onward at 8.5% with principal

t2_drawdown = 25_000_000
t2_start = date(2013, 12, 31)
t2_maturity = date(2022, 12, 31)
t2_io_rate = 0.09
t2_amort_rate = 0.085
t2_io_quarters = 20  # 5 years of IO
t2_amort_quarters = 16  # remaining 4 years
t2_principal_payment = t2_drawdown / t2_amort_quarters

print(f"Tranche 2 quarterly principal payment: ${t2_principal_payment:,.2f}")

t2_schedule = []
balance = 0
quarter_count = 0

for dt in all_dates:
    if dt < t2_start:
        continue
    if dt == t2_start:
        t2_schedule.append({
            'date': dt, 'opening': 0, 'drawdown': t2_drawdown,
            'interest_expense': 0, 'interest_paid': 0,
            'interest_capitalized': 0, 'principal_paid': 0,
            'closing': t2_drawdown
        })
        balance = t2_drawdown
        quarter_count = 0
        continue
    if dt > t2_maturity:
        break

    quarter_count += 1
    opening = balance

    if quarter_count <= t2_io_quarters:
        # Interest-only period at 9%
        interest = opening * t2_io_rate / 4
        principal = 0
    else:
        # Amortization period at 8.5%
        interest = opening * t2_amort_rate / 4
        principal = t2_principal_payment

    balance = opening - principal

    t2_schedule.append({
        'date': dt, 'opening': opening, 'drawdown': 0,
        'interest_expense': interest, 'interest_paid': interest,
        'interest_capitalized': 0, 'principal_paid': principal,
        'closing': balance
    })

print(f"Tranche 2 schedule has {len(t2_schedule)} entries")
print(f"Tranche 2 final closing balance: ${t2_schedule[-1]['closing']:,.6f}")

# Q3 check: debt service on 31 Dec 2020
for row in t2_schedule:
    if row['date'] == date(2020, 12, 31):
        ds = row['interest_paid'] + row['principal_paid']
        print(f"Tranche 2 debt service on 31 Dec 2020: ${ds:,.2f} = ${ds/1e6:.4f}m")
        print(f"  Interest: ${row['interest_paid']:,.2f}, Principal: ${row['principal_paid']:,.2f}")

In [ ]:
# ============================================================
# TRANCHE 3: Interest Only with Bullet at Maturity
# ============================================================
# $10m drawdown on 30 Jun 2014
# $4m additional drawdown on 31 Mar 2015
# 4 years from first drawdown => maturity 30 Jun 2018
# 7.45% p.a. throughout
# Interest paid every quarter, bullet payment at maturity

t3_start = date(2014, 6, 30)
t3_drawdown2_date = date(2015, 3, 31)
t3_maturity = date(2018, 6, 30)
t3_rate = 0.0745

t3_schedule = []
balance = 0

for dt in all_dates:
    if dt < t3_start:
        continue
    if dt > t3_maturity:
        break

    opening = balance
    drawdown = 0

    if dt == t3_start:
        drawdown = 10_000_000
    elif dt == t3_drawdown2_date:
        drawdown = 4_000_000

    # Interest calculated on opening balance (before drawdown)
    interest = opening * t3_rate / 4

    # Bullet payment at maturity
    if dt == t3_maturity:
        principal = opening + drawdown  # repay full balance
    else:
        principal = 0

    closing = opening + drawdown - principal
    balance = closing

    t3_schedule.append({
        'date': dt, 'opening': opening, 'drawdown': drawdown,
        'interest_expense': interest, 'interest_paid': interest,
        'interest_capitalized': 0, 'principal_paid': principal,
        'closing': closing
    })

total_interest_t3 = sum(r['interest_paid'] for r in t3_schedule)
print(f"Tranche 3 total interest paid: ${total_interest_t3:,.2f} = ${total_interest_t3/1e6:.4f}m")
print(f"Tranche 3 schedule has {len(t3_schedule)} entries")

In [ ]:
# ============================================================
# TRANCHE 4: Interest Capitalized then Level Principal
# ============================================================
# $22m drawdown on 30 Sep 2014
# 6 years => maturity 30 Sep 2020
# 8.20% for first 2 years (30 Sep 2014 to 30 Sep 2016)
# 7.70% for remainder
# Interest capitalized (added to balance) every quarter for 2 years
# Interest paid each quarter after that
# After capitalization period: 16 equal principal payments quarterly
# First payment on 31 December 2016

t4_drawdown_val = 22_000_000
t4_start = date(2014, 9, 30)
t4_maturity = date(2020, 9, 30)
t4_rate1 = 0.082   # first 2 years
t4_rate2 = 0.077   # remainder
t4_cap_end = date(2016, 9, 30)  # end of 2-year capitalization period
t4_first_principal = date(2016, 12, 31)

# Phase 1: Capitalization period
t4_schedule = []
balance = 0

for dt in all_dates:
    if dt < t4_start:
        continue
    if dt > t4_cap_end:
        break

    opening = balance
    drawdown = t4_drawdown_val if dt == t4_start else 0

    # Interest on opening balance at rate1
    interest = opening * t4_rate1 / 4
    # Interest is capitalized (not paid)
    closing = opening + drawdown + interest
    balance = closing

    t4_schedule.append({
        'date': dt, 'opening': opening, 'drawdown': drawdown,
        'interest_expense': interest, 'interest_paid': 0,
        'interest_capitalized': interest, 'principal_paid': 0,
        'closing': closing
    })

cap_end_balance = balance
print(f"Tranche 4 balance at end of cap period (30 Sep 2016): ${cap_end_balance:,.2f}")

# 16 equal principal payments
t4_principal_payment = cap_end_balance / 16
print(f"Tranche 4 quarterly principal payment: ${t4_principal_payment:,.2f}")

# Phase 2: Amortization period (31 Dec 2016 to 30 Sep 2020)
for dt in all_dates:
    if dt <= t4_cap_end:
        continue
    if dt > t4_maturity:
        break

    opening = balance
    interest = opening * t4_rate2 / 4
    principal = t4_principal_payment
    closing = opening - principal
    balance = closing

    t4_schedule.append({
        'date': dt, 'opening': opening, 'drawdown': 0,
        'interest_expense': interest, 'interest_paid': interest,
        'interest_capitalized': 0, 'principal_paid': principal,
        'closing': closing
    })

total_interest_t4 = sum(r['interest_expense'] for r in t4_schedule)
cap_interest_t4 = sum(r['interest_capitalized'] for r in t4_schedule)
print(f"Tranche 4 total interest expense: ${total_interest_t4:,.2f}")
print(f"Tranche 4 capitalized interest: ${cap_interest_t4:,.2f}")
print(f"Tranche 4 capitalized proportion: {cap_interest_t4/total_interest_t4*100:.1f}%")
print(f"Tranche 4 final closing balance: ${t4_schedule[-1]['closing']:,.6f}")

In [ ]:
# ============================================================
# ANSWER ALL QUESTIONS
# ============================================================

# Q1: Total interest paid on Tranche 3?
# Options: a. $3.9m  b. $6.2m  c. $4.6m  d. $3.5m
q1_total = sum(r['interest_paid'] for r in t3_schedule)
print(f"Q1: Total interest paid on Tranche 3: ${q1_total/1e6:.4f}m")
q1_options = {'A': 3.9e6, 'B': 6.2e6, 'C': 4.6e6, 'D': 3.5e6}
q1_answer = min(q1_options, key=lambda k: abs(q1_total - q1_options[k]))
print(f"  Closest option: {q1_answer}")

# Q2: Closing balance of Tranche 1 on 31 March 2020?
# Options: a. $1.5m  b. $1.7m  c. $1.9m  d. $2.1m
q2_val = None
for row in t1_schedule:
    if row['date'] == date(2020, 3, 31):
        q2_val = row['closing']
        break
print(f"\nQ2: Closing balance Tranche 1 on 31 Mar 2020: ${q2_val/1e6:.4f}m")
q2_options = {'A': 1.5e6, 'B': 1.7e6, 'C': 1.9e6, 'D': 2.1e6}
q2_answer = min(q2_options, key=lambda k: abs(q2_val - q2_options[k]))
print(f"  Closest option: {q2_answer}")

# Q3: Quarterly debt service for Tranche 2 on 31 Dec 2020?
# Options: a. $0.83m  b. $1.85m  c. $1.86m  d. $1.88m
q3_val = None
for row in t2_schedule:
    if row['date'] == date(2020, 12, 31):
        q3_val = row['interest_paid'] + row['principal_paid']
        break
print(f"\nQ3: Debt service Tranche 2 on 31 Dec 2020: ${q3_val/1e6:.4f}m")
q3_options = {'A': 0.83e6, 'B': 1.85e6, 'C': 1.86e6, 'D': 1.88e6}
q3_answer = min(q3_options, key=lambda k: abs(q3_val - q3_options[k]))
print(f"  Closest option: {q3_answer}")

# Q4: Proportion of Tranche 4 total interest expense that is capitalized?
# Options: a. 46.2%  b. 46.4%  c. 46.6%  d. 47.8%
q4_pct = cap_interest_t4 / total_interest_t4 * 100
print(f"\nQ4: Capitalized interest proportion: {q4_pct:.2f}%")
q4_options = {'A': 46.2, 'B': 46.4, 'C': 46.6, 'D': 47.8}
q4_answer = min(q4_options, key=lambda k: abs(q4_pct - q4_options[k]))
print(f"  Closest option: {q4_answer}")

# Q5: Total interest expense for entire facility in calendar year 2016?
# Options: a. $6.07m  b. $6.11m  c. $6.15m  d. $6.19m
q5_total = 0
for schedule in [t1_schedule, t2_schedule, t3_schedule, t4_schedule]:
    for row in schedule:
        if row['date'].year == 2016:
            q5_total += row['interest_expense']
print(f"\nQ5: Total interest expense in 2016: ${q5_total/1e6:.4f}m")
q5_options = {'A': 6.07e6, 'B': 6.11e6, 'C': 6.15e6, 'D': 6.19e6}
q5_answer = min(q5_options, key=lambda k: abs(q5_total - q5_options[k]))
print(f"  Closest option: {q5_answer}")

# Q6: When does facility closing balance drop below $40m?
# Options: a. 30 Sep 2018  b. 31 Dec 2018  c. 31 Mar 2019  d. 30 Jun 2020
# Combine all schedules and compute total closing balance per date
balance_by_date = {}
for schedule in [t1_schedule, t2_schedule, t3_schedule, t4_schedule]:
    for row in schedule:
        dt = row['date']
        balance_by_date[dt] = balance_by_date.get(dt, 0) + row['closing']

sorted_dates = sorted(balance_by_date.keys())
q6_date = None
prev_bal = None
print(f"\nQ6: Facility closing balance by date:")
for dt in sorted_dates:
    bal = balance_by_date[dt]
    if dt >= date(2017, 1, 1) and dt <= date(2020, 12, 31):
        marker = " *** BELOW 40m" if bal < 40e6 else ""
        print(f"  {dt}: ${bal/1e6:.4f}m{marker}")
    if prev_bal is not None and prev_bal >= 40e6 and bal < 40e6 and q6_date is None:
        q6_date = dt
    prev_bal = bal

print(f"\n  Facility drops below $40m on: {q6_date}")
q6_options = {
    'A': date(2018, 9, 30),
    'B': date(2018, 12, 31),
    'C': date(2019, 3, 31),
    'D': date(2020, 6, 30)
}
q6_answer = None
for key, val in q6_options.items():
    if val == q6_date:
        q6_answer = key
        break
if q6_answer is None:
    # Find closest
    q6_answer = min(q6_options, key=lambda k: abs((q6_date - q6_options[k]).days))
print(f"  Answer: {q6_answer}")

In [ ]:
# ============================================================
# FINAL ANSWERS
# ============================================================
answers = {
    "question1": q1_answer,
    "question2": q2_answer,
    "question3": q3_answer,
    "question4": q4_answer,
    "question5": q5_answer,
    "question6": q6_answer,
}

print("Final answers:")
for k, v in answers.items():
    print(f"  {k}: {v}")